# Intermediate 05 — Moment Generating and Characteristic Functions

Practice notebook for the **"Moment Generating and Characteristic Functions"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Moment generating functions: definition and Monte Carlo check

The **moment generating function** (mgf) of a random variable \(X\) is

$$
M_X(t) = E\left[e^{tX}\right],
$$

defined for all \(t\) where the expectation is finite. If \(M_X\) exists in an
open interval around 0, it **uniquely determines the distribution** of \(X\).
Moments follow by differentiation:

$$
M_X^{(k)}(0) = E\left[X^k\right].
$$

For \(X \sim N(\mu, \sigma^2)\) the closed form is

$$
M_X(t) = \exp\left(\mu t + \tfrac{1}{2}\sigma^2 t^2\right).
$$

Below we estimate \(M_X(t) = E[e^{tX}]\) by Monte Carlo for a normal and an
exponential, and overlay the closed-form curves.


In [ ]:
# Monte Carlo MGF vs closed form for Normal and Exponential
rng = np.random.default_rng(42)
N = 200_000

# --- Normal: X ~ N(mu, sigma^2) ---
mu, sigma = 1.0, 2.0
Xn = rng.normal(mu, sigma, size=N)
t = np.linspace(-0.6, 0.6, 121)
mgf_norm_mc = np.mean(np.exp(np.outer(Xn, t)), axis=0)
mgf_norm_cf = np.exp(mu * t + 0.5 * sigma**2 * t**2)

# --- Exponential: Y ~ Exp(lambda) ---
lam = 1.5
Ye = rng.exponential(1.0 / lam, size=N)
# Closed form MGF exists for t < lambda: M(t) = lambda / (lambda - t)
mgf_exp_mc = np.mean(np.exp(np.outer(Ye, t[t < lam])), axis=0)
mgf_exp_cf = lam / (lam - t[t < lam])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(t, mgf_norm_mc, label="Monte Carlo", lw=2)
ax[0].plot(t, mgf_norm_cf, "k--", label="Closed form", lw=1.5)
ax[0].set_title(f"MGF of N({mu}, {sigma**2})")
ax[0].set_xlabel("t"); ax[0].set_ylabel(r"$M_X(t)$"); ax[0].legend()

ax[1].plot(t[t < lam], mgf_exp_mc, label="Monte Carlo", lw=2)
ax[1].plot(t[t < lam], mgf_exp_cf, "k--", label="Closed form", lw=1.5)
ax[1].set_title(f"MGF of Exp({lam})  (exists for t < {lam})")
ax[1].set_xlabel("t"); ax[1].set_ylabel(r"$M_X(t)$"); ax[1].legend()
plt.tight_layout(); plt.show()

# Max abs error
err_norm = np.max(np.abs(mgf_norm_mc - mgf_norm_cf))
err_exp  = np.max(np.abs(mgf_exp_mc - mgf_exp_cf))
print(f"Normal  max |MC - CF| = {err_norm:.4f}")
print(f"Exponential max |MC - CF| = {err_exp:.4f}")


**Your turn:** try \(t\) values closer to the boundary \(\lambda\) for the
exponential — why does the Monte Carlo estimate blow up?


---
## Part 2 — Recovering moments by differentiating the MGF

Because \(M_X^{(k)}(0) = E[X^k]\), we can recover moments by differentiating the MGF.
We use **finite differences** numerically, and (if available) **sympy** symbolically.

For the normal, \(M_X'(0) = \mu\) and \(M_X''(0) = \mu^2 + \sigma^2\), giving
\(\operatorname{Var}(X) = M_X''(0) - (M_X'(0))^2 = \sigma^2\).


In [ ]:
# Recover moments of N(mu, sigma^2) from the closed-form MGF via finite differences
mu, sigma = 1.0, 2.0
M = lambda s: np.exp(mu * s + 0.5 * sigma**2 * s**2)

# Central finite-difference derivatives at s = 0
h = 1e-4
M1 = (M(h) - M(-h)) / (2 * h)                       # M'(0)  = E[X]
M2 = (M(h) - 2 * M(0) + M(-h)) / (h**2)             # M''(0) = E[X^2]
var_fd = M2 - M1**2

print(f"E[X]   finite-diff = {M1:.6f}   (exact mu = {mu})")
print(f"E[X^2] finite-diff = {M2:.6f}   (exact = {mu**2 + sigma**2})")
print(f"Var(X) finite-diff = {var_fd:.6f}   (exact sigma^2 = {sigma**2})")

# Same idea, but sampling-based: differentiate the *empirical* MGF
rng = np.random.default_rng(42)
N = 500_000
X = rng.normal(mu, sigma, size=N)
Memp = lambda s: np.mean(np.exp(s * X))
M1e = (Memp(h) - Memp(-h)) / (2 * h)
M2e = (Memp(h) - 2 * Memp(0) + Memp(-h)) / (h**2)
print(f"\nFrom empirical MGF:")
print(f"E[X]   = {M1e:.6f}   Var(X) = {M2e - M1e**2:.6f}")


In [ ]:
# Symbolic differentiation with sympy (optional — only if importable)
try:
    import sympy as sp
    s, mu_s, sig_s = sp.symbols("s mu sigma", real=True)
    M_sym = sp.exp(mu_s * s + sp.Rational(1, 2) * sig_s**2 * s**2)
    d1 = sp.diff(M_sym, s).subs(s, 0).simplify()
    d2 = sp.diff(M_sym, s, 2).subs(s, 0).simplify()
    print("sympy M'(0)  =", d1)
    print("sympy M''(0) =", d2)
    print("Var(X)       =", sp.simplify(d2 - d1**2))
except ImportError:
    print("sympy not available — skipped symbolic differentiation.")


**Your turn:** recover the first three moments of an \(\operatorname{Exp}(\lambda)\)
distribution from its MGF \(\lambda/(\lambda - t)\).


---
## Part 3 — Characteristic functions

The **characteristic function** of \(X\) is

$$
\varphi_X(t) = E\left[e^{itX}\right], \quad t \in \mathbb{R}.
$$

Characteristic functions **always exist** (\(|\varphi_X(t)| \leq 1\)) and uniquely
determine the distribution. They **linearize sums**: if \(X \perp Y\),

$$
\varphi_{X+Y}(t) = \varphi_X(t)\,\varphi_Y(t).
$$

For \(X \sim N(\mu, \sigma^2)\),

$$
\varphi_X(t) = \exp\left(i\mu t - \tfrac{1}{2}\sigma^2 t^2\right).
$$

We estimate \(\varphi_X(t)\) by Monte Carlo and compare real/imaginary parts to
the closed form.


In [ ]:
# Empirical vs theoretical characteristic function for a normal
rng = np.random.default_rng(42)
N = 300_000
mu, sigma = 0.5, 1.5
X = rng.normal(mu, sigma, size=N)

t = np.linspace(-4, 4, 201)
# Empirical CF: average of e^{i t X} -> complex mean over samples
phi_emp = np.mean(np.exp(1j * np.outer(X, t)), axis=0)
# Theoretical
phi_cf = np.exp(1j * mu * t - 0.5 * sigma**2 * t**2)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(t, phi_emp.real, label="MC real", lw=2)
ax[0].plot(t, phi_cf.real, "k--", label="Theory real", lw=1.5)
ax[0].plot(t, phi_emp.imag, label="MC imag", lw=2, alpha=0.8)
ax[0].plot(t, phi_cf.imag, "r--", label="Theory imag", lw=1.5)
ax[0].set_title(rf"Characteristic function of N({mu}, {sigma**2})")
ax[0].set_xlabel("t"); ax[0].set_ylabel(r"$\varphi_X(t)$"); ax[0].legend()

ax[1].plot(t, np.abs(phi_emp), label="MC |phi|", lw=2)
ax[1].plot(t, np.abs(phi_cf), "k--", label="Theory |phi|", lw=1.5)
ax[1].set_title("Modulus (bounded by 1)")
ax[1].set_xlabel("t"); ax[1].legend()
plt.tight_layout(); plt.show()

err = np.max(np.abs(phi_emp - phi_cf))
print(f"max |phi_MC - phi_theory| = {err:.5f}")
print(f"max |phi_MC| (should be <= 1) = {np.max(np.abs(phi_emp)):.5f}")


**Your turn:** verify numerically that \(|\varphi_X(t)| \leq 1\) for all \(t\),
even for a heavy-tailed distribution (try a Cauchy).


---
## Part 4 — Linearization of sums: independent normals

If \(X \sim N(\mu_1, \sigma_1^2)\) and \(Y \sim N(\mu_2, \sigma_2^2)\) are
independent, then \(\varphi_{X+Y}(t) = \varphi_X(t)\varphi_Y(t)\), which is the CF
of \(N(\mu_1+\mu_2,\ \sigma_1^2+\sigma_2^2)\). We verify the product rule by
Monte Carlo: the empirical CF of \(X+Y\) should match the product of the marginal CFs.


In [ ]:
# CF of a sum = product of CFs (independent normals)
rng = np.random.default_rng(42)
N = 300_000
mu1, s1 = 1.0, 2.0
mu2, s2 = -0.5, 1.0
X = rng.normal(mu1, s1, size=N)
Y = rng.normal(mu2, s2, size=N)
S = X + Y

t = np.linspace(-3, 3, 201)
phi_X = np.exp(1j * mu1 * t - 0.5 * s1**2 * t**2)
phi_Y = np.exp(1j * mu2 * t - 0.5 * s2**2 * t**2)
phi_prod = phi_X * phi_Y            # theory: product of marginal CFs
phi_S_mc = np.mean(np.exp(1j * np.outer(S, t)), axis=0)   # empirical CF of S
phi_S_theory = np.exp(1j * (mu1 + mu2) * t - 0.5 * (s1**2 + s2**2) * t**2)

plt.figure(figsize=(9, 5))
plt.plot(t, phi_S_mc.real, lw=2, label="MC CF of X+Y (real)")
plt.plot(t, phi_prod.real, "k--", lw=1.5, label="phi_X * phi_Y (real)")
plt.plot(t, phi_S_mc.imag, lw=2, alpha=0.8, label="MC CF of X+Y (imag)")
plt.plot(t, phi_prod.imag, "r--", lw=1.5, label="phi_X * phi_Y (imag)")
plt.title("Sum of independent normals: CF factorizes")
plt.xlabel("t"); plt.legend()
plt.tight_layout(); plt.show()

err1 = np.max(np.abs(phi_S_mc - phi_prod))
err2 = np.max(np.abs(phi_S_mc - phi_S_theory))
print(f"max |phi_S_MC - phi_X*phi_Y|        = {err1:.5f}")
print(f"max |phi_S_MC - N(mu1+mu2, sig1+sig2) CF| = {err2:.5f}")
print(f"Implied sum distribution: N({mu1+mu2}, {s1**2 + s2**2})")


**Your turn:** repeat with \(X\) normal and \(Y\) exponential (independent).
The product \(\varphi_X(t)\varphi_Y(t)\) should still match the empirical CF of \(X+Y\),
even though the sum is no longer normal.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Compute the MGF of \(X \sim \operatorname{Exp}(\lambda)\) by integration and state the domain of \(t\) where it is finite. Then verify \(M_X'(0)=1/\lambda\) and \(M_X''(0)=2/\lambda^2\) numerically.

2. Let \(X \sim N(2, 9)\). Using the closed-form MGF, compute \(E[X]\), \(E[X^2]\) and \(\operatorname{Var}(X)\) by finite-difference differentiation at \(t=0\). Confirm \(\operatorname{Var}(X)=9\).

3. Estimate the characteristic function of a standard Cauchy by Monte Carlo. The closed form is \(\varphi(t)=e^{-|t|}\). Plot the empirical modulus and confirm \(|\varphi(t)|\leq 1\) everywhere.

4. If \(X\sim N(1,4)\) and \(Y\sim N(3,1)\) are independent, use the factorization \(\varphi_{X+Y}=\varphi_X\varphi_Y\) to identify the distribution of \(X+Y\). Verify numerically that the empirical CF of \(X+Y\) matches \(\varphi_X\varphi_Y\).

5. Show via the empirical MGF that \(M_X(t)\) for an exponential diverges as \(t \uparrow \lambda\). Plot \(M_X(t)\) against its closed form on \(t \in [0, 0.95\lambda]\) and report where the Monte Carlo estimate first departs from the closed form by more than 5%.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(M_X(t)=\int_0^\infty e^{tx}\lambda e^{-\lambda x}\,dx = \frac{\lambda}{\lambda-t}\) for \(t<\lambda\).
\(M'(t)=\frac{\lambda}{(\lambda-t)^2}\Rightarrow M'(0)=1/\lambda\);
\(M''(t)=\frac{2\lambda}{(\lambda-t)^3}\Rightarrow M''(0)=2/\lambda^2\). Finite-difference at \(t=0\) with small \(h\) reproduces these to several digits.

**2.** \(M(t)=e^{2t+4.5t^2}\). \(M'(0)=2=E[X]\); \(M''(0)=2^2+9=13=E[X^2]\); \(\operatorname{Var}(X)=13-4=9\). Code:
```python
import numpy as np
M = lambda t: np.exp(2*t + 4.5*t**2); h = 1e-4
E1 = (M(h)-M(-h))/(2*h); E2 = (M(h)-2*M(0)+M(-h))/h**2
print(E1, E2, E2 - E1**2)   # ~2.0 13.0 9.0
```

**3.** Cauchy has no mean but its CF exists:
```python
rng = np.random.default_rng(42)
X = rng.standard_cauchy(500_000)
t = np.linspace(-5, 5, 201)
phi = np.mean(np.exp(1j*np.outer(X, t)), axis=0)
print(np.max(np.abs(phi)))   # <= 1
plt.plot(t, np.abs(phi), label="MC"); plt.plot(t, np.exp(-np.abs(t)), "k--", label="theory")
```
The modulus stays \(\leq 1\), consistent with \(e^{-|t|}\).

**4.** \(\varphi_X(t)=e^{it-2t^2}\), \(\varphi_Y(t)=e^{3it-0.5t^2}\), product \(=e^{4it-2.5t^2}\), the CF of \(N(4, 5)\). So \(X+Y\sim N(4,5)\). The empirical CF of \(X+Y\) overlays \(\varphi_X\varphi_Y\) to within Monte Carlo error.

**5.** As \(t\to\lambda^-\), \(\lambda/(\lambda-t)\to\infty\). Monte Carlo with finite \(N\) averages \(e^{tX}\); once \(t\gtrsim 0.9\lambda\) a few large draws dominate and the estimate becomes unstable, typically departing by >5% around \(t\approx 0.9\lambda\) for \(N=10^5\).

</details>
